> 💻 **This notebook runs locally — embedded mode.** `pip install figuard`; enforcement runs in-process against a local SQLite file (`~/.figuard/figuard.db`). No server, no API key, no network. Same rules as the server, held in lockstep by a shared conformance suite.
>
> **Three ways to run FiGuard — same API, same rules. Swap only the client constructor:**
>
> - 💻 **Embedded** _(this notebook)_ — `FiGuardClient()` — one app/agent, zero infra.
> - ☁️ **Free sandbox** — `FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")` — quick hosted evaluation (not for production; may be wiped; no SLA).
> - 🏢 **Self-hosted server** — `FiGuardClient(base_url="https://your-figuard", api_key="...")` — budgets shared across many agents/processes; your data, your infra. [Self-host →](https://github.com/figuard/figuard-core#self-hosting)
>
> Fleet features — delegation, category allocations, shadow mode, cross-process budgets — require the sandbox or a server.

# Incident 2 — The Duplicate Invoice Payment

**What happened:** An AP agent processed an invoice for $1,500. A network timeout on the first attempt meant the agent never received the confirmation. It retried. The same invoice was paid twice. Finance found the $1,500 duplicate **three weeks later** during reconciliation. Dispute resolution took another two weeks.

**Why it happens:** Retries create new payment intents. Without idempotency the processor has no way to know it has already seen this request.

**The fix:** An idempotency key tied to the invoice ID. FiGuard deduplicates at the authorization layer — a retry with the same key returns the original event instead of creating a new reservation. The payment processor is called exactly once.

In [ ]:
# @title Step 1 — Install and configure FiGuard
import sys
!pip install "figuard>=0.3.2" --upgrade -q
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")
import uuid
USER_ID = f"demo_{uuid.uuid4().hex[:8]}"
print(f"Your session ID: {USER_ID}  (labels your budgets locally)")


## Without FiGuard — two charges, one invoice

Each call creates a new charge ID. The retry produces a second charge.

In [ ]:
# WITHOUT FiGuard — no idempotency

def process_payment(invoice_id, amount, attempt):
    # Simulated payment processor — each call creates a new charge
    return {"id": f"ch_{invoice_id}_{attempt}_{'x'*8}", "amount": amount}

invoice_id = "INV-2026-0342"
amount = 1500.00

print(f"Processing {invoice_id} for ${amount:.2f}")
print()

charge1 = process_payment(invoice_id, amount, attempt=1)
print(f"Attempt 1: charge {charge1['id']}")
print(f"           ← network timeout — agent never got this response")
print()

charge2 = process_payment(invoice_id, amount, attempt=2)
print(f"Attempt 2: charge {charge2['id']}")
print()

print(f"Two different charge IDs: {charge1['id'] != charge2['id']}")
print(f"Amount charged: ${amount * 2:.2f}  (expected ${amount:.2f})")
print(f"Duplicate discovered: 3 weeks later")

## With FiGuard — idempotency key deduplicates the retry

The retry uses the same idempotency key. FiGuard returns the original authorization event — same `event_id`, one reservation, one confirmation.

In [ ]:
# Embedded mode has no web UI — get_spend_tree() *is* the spend view (the same data a dashboard renders).
def show_spend_tree(client, budget_id):
    b = client.get_budget(budget_id)
    tree = client.get_spend_tree(budget_id)
    unit = b.currency or b.unit or ""
    print(f"\nSpend tree — {b.quantity_spent}/{b.total_limit} {unit} spent, "
          f"{b.available_quantity} available  ({tree.total_events} events)")
    glyph = {"CONFIRMED": "✓", "AUTHORIZED": "⏳", "DENIED": "✗", "FAILED": "⚠", "VOIDED": "↩"}
    def walk(node, depth=1):
        e = node.event
        amt = e.confirmed_quantity if e.confirmed_quantity is not None else e.requested_quantity
        label = e.description or e.entity_id or e.action_type
        label = f"{label} — " if label else ""
        print("   " * depth + f"{glyph.get(e.decision, '•')} {label}{amt} {e.currency or ''} [{e.decision}]")
        for ch in node.children:
            walk(ch, depth + 1)
    for r in tree.roots:
        walk(r)


In [ ]:
from figuard import FiGuardClient

client = FiGuardClient()

budget = client.create_budget(
    user_id=USER_ID,
    total_limit=50_000.00,
    currency="USD",
    expires_in="1h",
    entity_dedup_enabled=True,
)

session_token = budget.session_token


invoice_id = "INV-2026-0342"
amount = 1500.00

print(f"Processing {invoice_id} for ${amount:.2f}")
print()

# Attempt 1 — authorized, response lost
auth1 = client.authorize(
    session_token=session_token,
    agent_id="ap_agent",
    action_type="PAYMENT",
    description=f"Vendor payment {invoice_id}",
    requested_quantity=amount,
    idempotency_key=f"invoice-{invoice_id}",
)
print(f"Attempt 1: {auth1.decision} — event {auth1.event_id}")
print(f"           ← network timeout")
print()

# Retry — same key returns original event
auth2 = client.authorize(
    session_token=session_token,
    agent_id="ap_agent",
    action_type="PAYMENT",
    description=f"Vendor payment {invoice_id}",
    requested_quantity=amount,
    idempotency_key=f"invoice-{invoice_id}",
)
print(f"Attempt 2: {auth2.decision} — event {auth2.event_id}")
print()

print(f"✓ Same event returned: {auth1.event_id == auth2.event_id}")
print(f"  Amount authorized:   ${amount:.2f}  (not ${amount * 2:.2f})")

if auth1.is_authorized:
    client.confirm_event(auth1.event_id, confirmed_quantity=amount)
    print(f"  Confirmed once:      ${amount:.2f}")

show_spend_tree(client, budget.id)